# Latest Code

## Importing Libraries and Loading env

In [1]:
import json
import os
import time

from dotenv import load_dotenv
from datasets import Dataset

from rag_eval_adapter import query_rag

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_similarity,
    answer_correctness,
    context_precision,
    context_recall,
)

from ragas.llms import llm_factory

from openai import OpenAI
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from sentence_transformers import SentenceTransformer


load_dotenv()

c:\Users\Nakul\anaconda3\envs\ragas_eval_py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\949025462.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\949025462.py:12: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\949025462.py:12: DeprecationWarning: Importing a

True

## Hugging Face Authentication

In [2]:
# Hugging Face Authentication
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    print("✅ Hugging Face Token Loaded")
else:
    print("⚠️ HF_TOKEN not found")

✅ Hugging Face Token Loaded


## Configure environment variables

In [3]:
BENCHMARK_FILE = "rag_benchmark_50q.json"
CACHE_FILE = "rag_outputs.json"
RESULTS_FILE = "evaluation_results.json"
GEN_TIME_FILE = "generation_times.json"
JUDGE_MODEL ="nvidia/nemotron-3-ultra-550b-a55b:free"

## Ollama Judge

In [4]:
judge_llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0,
    num_ctx=8192,
)

ragas_llm = LangchainLLMWrapper(judge_llm)
print("Judge Model:", judge_llm.model)

Judge Model: qwen2.5:7b


C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\1801076943.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(judge_llm)


## Embedding Model(Ollama)

In [5]:
OllamaEmbeddings(
    model="nomic-embed-text-v2-moe:latest"
)

OllamaEmbeddings(model='nomic-embed-text-v2-moe:latest', base_url=None, client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## Embedding Model(Hugging Face)

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = LangchainEmbeddingsWrapper(hf_embeddings)

C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\1733445127.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3332.59it/s]
C:\Users\Nakul\AppData\Local\Temp\ipykernel_24096\1733445127.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embedding_model = LangchainEmbeddingsWrapper(hf_embed

## Build / Load Dataset

In [7]:
if os.path.exists(CACHE_FILE):

    print(f"Loading cached outputs from {CACHE_FILE}")

    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        evaluation_rows = json.load(f)

    if os.path.exists(GEN_TIME_FILE):
        with open(GEN_TIME_FILE, "r", encoding="utf-8") as f:
            generation_times = json.load(f)

        print(
            f"Loaded {len(generation_times)} cached generation times"
        )

else:

    print("Generating RAG outputs...")

    evaluation_rows = []
    generation_times = []
    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)

    for idx, sample in enumerate(benchmark, start=1):

        question = sample["question"]

        print(f"\n========== QUESTION {idx} ==========")
        print(question)

        print(f"[{idx}/{len(benchmark)}] Processing")

        start = time.time()

        rag_answer, retrieved_contexts, _ = query_rag(question)

        generation_time = time.time() - start
        generation_times.append(generation_time)

        print(
            f"QUESTION {idx} COMPLETED "
            f"({generation_time:.2f} sec)"
        )

        evaluation_rows.append(
            {
                "user_input": question,
                "response": rag_answer,
                "retrieved_contexts": retrieved_contexts,
                "reference": sample["reference_answer"],
            }
        )

        # Save after every question
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(
                evaluation_rows,
                f,
                indent=2,
                ensure_ascii=False,
            )

        with open(GEN_TIME_FILE, "w", encoding="utf-8") as f:
            json.dump(
                generation_times,
                f,
                indent=2,
            )

    print(f"Saved outputs to {CACHE_FILE}")
    print(f"Saved generation times to {GEN_TIME_FILE}")

Generating RAG outputs...

========== QUESTION 1 ==========
What BLEU score did the Transformer (big) model achieve on the WMT 2014 English-to-German translation task, and by how much did it outperform the previous best results including ensembles?
[1/50] Processing
Persist Directory: C:\Users\Nakul\Documents\rag_rag\Multimodal-RAG\backend\chroma_db
Collection Name: langchain
Collection Count: 70
ABSOLUTE DB PATH: C:\Users\Nakul\Documents\rag_rag\Multimodal-RAG\backend\chroma_db
Collection Count: 70
START RETRIEVAL
RETRIEVAL DONE
Retrieval took 0.21s
Candidate Chunks Retrieved: 10
Candidate Chunk 1: 1693 chars
Candidate Chunk 2: 1708 chars
Candidate Chunk 3: 325 chars
Candidate Chunk 4: 268 chars
Candidate Chunk 5: 625 chars
Candidate Chunk 6: 325 chars
Candidate Chunk 7: 2281 chars
Candidate Chunk 8: 1268 chars
Candidate Chunk 9: 558 chars
Candidate Chunk 10: 325 chars


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4324.20it/s]


RERANK DONE
Rerank took 10.88s
Prompt Length: 8665
Approx Tokens: 2166
Prompt Length: 8771
Approx Tokens: 2192
GENERATION DONE
Generation took 10.05s
QUESTION 1 COMPLETED (21.37 sec)

========== QUESTION 2 ==========
How many identical layers compose the Transformer encoder stack, and what are the two sub-layers within each encoder layer?
[2/50] Processing
Collection Count: 70
START RETRIEVAL
RETRIEVAL DONE
Retrieval took 4.06s
Candidate Chunks Retrieved: 10
Candidate Chunk 1: 1349 chars
Candidate Chunk 2: 1347 chars
Candidate Chunk 3: 657 chars
Candidate Chunk 4: 812 chars
Candidate Chunk 5: 2845 chars
Candidate Chunk 6: 325 chars
Candidate Chunk 7: 2475 chars
Candidate Chunk 8: 325 chars
Candidate Chunk 9: 504 chars
Candidate Chunk 10: 625 chars
RERANK DONE
Rerank took 3.13s
Prompt Length: 7207
Approx Tokens: 1801
Prompt Length: 7285
Approx Tokens: 1821
GENERATION DONE
Generation took 13.05s
QUESTION 2 COMPLETED (20.24 sec)

========== QUESTION 3 ==========
What additional (third) su

## Dataset

In [8]:
eval_dataset = Dataset.from_list(evaluation_rows)

print(f"Dataset Size: {len(eval_dataset)}")

Dataset Size: 50


## Metric Configuration

In [9]:
faithfulness.llm = ragas_llm
context_precision.llm = ragas_llm
context_recall.llm = ragas_llm

answer_relevancy.embeddings = embedding_model

answer_similarity.embeddings = embedding_model

answer_correctness.llm = ragas_llm
answer_correctness.embeddings = embedding_model

metrics = [
    faithfulness,
    answer_relevancy,
    answer_similarity,
    answer_correctness,
    context_precision,
    context_recall,
]

## Evaluation

In [10]:
print("\nStarting Evaluation...\n")

eval_start = time.time()

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    batch_size=1,
)

eval_time = time.time() - eval_start


Starting Evaluation...



Evaluating:  56%|█████▌    | 168/300 [36:19<30:30, 13.87s/it]Exception raised in Job[168]: OutputParserException(Invalid json output: {"statements": [{"statement": "According to the ablation study (Table 3, row B), reducing the attention key size \\( dk \\) hurts model quality.","reason": "The context mentions that in Table 3 rows (B), reducing the attention key size \\( dk \\) hurts model quality. This directly supports the statement.","verdict": 1}, {"statement": "Reducing the attention key size \\( dk \\) suggests that determining compatibility is not easy.","reason": "The context states, 'This suggests that determining compatibility is not easy and a more sophisticated compatibility function than dot product may be beneficial.' This directly supports the statement.","verdict": 1}, {"statement": "Determining compatibility is not easy and a more sophisticated compatibility function than dot product may be beneficial.","reason": "The context explicitly states, 'This suggests that dete

## Results

In [11]:
results_dict = results.to_pandas().mean(numeric_only=True).to_dict()

if "faithfulness" in results_dict:
    results_dict["hallucination_rate"] = (
        1 - results_dict["faithfulness"]
    )

if generation_times:
    results_dict["avg_generation_latency_sec"] = (
        sum(generation_times) / len(generation_times)
    )
    results_dict["total_generation_time_sec"] = (
    sum(generation_times)
    )

results_dict["evaluation_time_sec"] = eval_time

print("\n===== RESULTS =====")

for k, v in results_dict.items():
    print(f"{k}: {v}")
    



===== RESULTS =====
faithfulness: 0.8441205648926237
answer_relevancy: 0.8164406202275352
answer_similarity: 0.8010881223030947
answer_correctness: 0.7186573535860965
context_precision: 0.8939342403177178
context_recall: 0.851
hallucination_rate: 0.15587943510737634
avg_generation_latency_sec: 23.575678935050963
total_generation_time_sec: 1178.7839467525482
evaluation_time_sec: 4786.143550395966


## Save Results

In [12]:
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(
        results_dict,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"\nSaved results to {RESULTS_FILE}")


Saved results to evaluation_results.json
